# Customer Bronze table ingestion

Full-load pipeline for `retail_analytics.bronze.customers`: the **most recently modified** CSV dropped in the customers volume becomes the source and fully overwrites the bronze table.

**Flow:** 1. Environment setup → 2. Read the latest file → 3. Materialize the bronze table → 4. Table maintenance → 5. Run history

## 1. Environment setup

Creates the `retail_analytics` catalog, the `bronze` schema, and the `raw_data` volume that holds the incoming CSV files. All statements are idempotent (`IF NOT EXISTS`), so the cell is safe to re-run.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail_analytics;
CREATE SCHEMA IF NOT EXISTS retail_analytics.bronze;
CREATE VOLUME IF NOT EXISTS retail_analytics.bronze.raw_data

## 2. Full load: read the latest file

Lists all `customers*.csv` files in the raw-data volume and picks the **most recently modified** one, so any newly dropped file automatically becomes the ingestion source. The file is read with a **static schema** (`registration_date` and `updated_at` parsed as `DateType`) and enriched with lineage metadata:

* `_ingested_at` — load timestamp
* `_source_file` — path of the file loaded
* `_batch_id` — unique run identifier

In [0]:
import uuid
from pyspark.sql.functions import current_timestamp, lit

# Configs
source_dir = '/Volumes/retail_analytics/bronze/raw_data/customers/'
has_header = 'true'
batch_id = str(uuid.uuid4())

# Full load: pick the most recently modified CSV file in the source directory
files = [retail_analytics.bronze.customers_ingestion_log
    f for f in dbutils.fs.ls(source_dir)
    if f.name.lower().endswith('.csv') and f.name.startswith('customers')
]
if not files:
    raise ValueError(f'No CSV files found in {source_dir}')

latest_file = max(files, key=lambda f: f.modificationTime)
file_path = latest_file.path.replace('dbfs:', '')  # normalize to /Volumes/... form

# Defining static schema
from pyspark.sql.types import StructType, StructField, StringType, DateType

schema = StructType([
  StructField("customer_id", StringType(), True),
  StructField("customer_name", StringType(), True),
  StructField("email", StringType(), True),
  StructField("registration_date", DateType(), True),
  StructField("city", StringType(), True),
  StructField("state", StringType(), True),
  StructField("customer_segment", StringType(), True),
  StructField("updated_at", DateType(), True),
  StructField("source_system", StringType(), True),
])

# Reading the data (static schema applied via .schema(), not an option)
df = spark.read \
    .format('csv') \
    .option('header', has_header) \
    .schema(schema) \
    .load(file_path)

# Creating 
df = df.withColumn("_ingested_at", current_timestamp())
df = df.withColumn("_source_file", lit(file_path))
df = df.withColumn("_batch_id", lit(batch_id))

print(f'Full load from latest file: {file_path}')
df.show(10)

## 3. Materialize the bronze table

Writes the DataFrame to `retail_analytics.bronze.customers` in **overwrite** mode — full-load semantics: the entire table is replaced by the latest file's rows. `overwriteSchema` lets the write succeed even if the table schema needs to change.

In [0]:
# Configs
write_mode = 'overwrite'
mat_mode = 'delta'

# Materializing the bronze table (full load: entire table is replaced by the latest file)
df.write.format(mat_mode).mode(write_mode).option('overwriteSchema', 'true').saveAsTable('retail_analytics.bronze.customers')

## 4. Table maintenance

* **VACUUM** removes Delta data files older than 168 hours (7 days). ⚠️ Time travel beyond the retention window is permanently lost, so run this deliberately, not on every execution.
* **OPTIMIZE ... ZORDER BY (`customer_id`)** compacts small files and physically colocates rows by `customer_id`, improving read performance for queries and joins that filter on it.

In [0]:
%sql
-- Vacuum table for optmization
VACUUM retail_analytics.bronze.customers RETAIN 168 HOURS;


In [0]:
%sql
-- ZORDER
OPTIMIZE retail_analytics.bronze.customers
ZORDER by (`customer_ID`);

## 5. Run history

`DESCRIBE HISTORY` shows every version of the bronze table — each full-load overwrite creates a new version with its timestamp and operation details. Useful for auditing and time travel (`SELECT * FROM ... VERSION AS OF <version>`).

In [0]:
%sql
describe history retail_analytics.bronze.customers